In [ ]:
!pip install gensim bertopic

In [1]:
import pickle
with open('/content/drive/MyDrive/Bertopic/Data/bs_sen.pkl', 'rb') as f:
    sentences = pickle.load(f)

In [2]:
print(len(sentences))

120881


In [ ]:
# Compute the term freqency dictionary
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Step 1: Create and fit the vectorizer
vectorizer = CountVectorizer(
    stop_words="english",
    min_df=2,
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w{3,}\b"
)

X = vectorizer.fit_transform(sentences)

# Step 2: Get token names and their total frequencies
terms = vectorizer.get_feature_names_out()
frequencies = np.asarray(X.sum(axis=0)).flatten()

# Step 3: Create the dictionary
term_freq_dict = dict(zip(terms, frequencies))
total_term_freq = sum(frequencies)
# Example: print the top 10 most frequent terms
top_10 = sorted(term_freq_dict.items(), key=lambda x: x[1], reverse=True)[:10]
print("🔝 Top 10 terms by frequency:")
for term, freq in top_10:
    print(f"{term:<20} : {freq}")


🔝 Top 10 terms by frequency:
trade                : 6751
union                : 5858
members              : 5559
work                 : 4551
labour               : 4476
time                 : 4048
general              : 3341
council              : 3278
workers              : 2803
conference           : 2462


In [ ]:
import random
import copy
import numpy as np
import pandas as pd
import ast
import re
from bertopic import BERTopic
from umap.umap_ import UMAP
from hdbscan import HDBSCAN
import gc
import time

error_sizes = []
topic20_sizes = []
gini_scores = []
ngram_values = []
silhouettes= []
coherences = []
model_names = []
time_costs = []

In [ ]:
# You will need to pre-computer your vector models and store them prior to this

dir = '/content/drive/MyDrive/V_model'
import os
list_of_files = os.listdir(dir)
print(list_of_files)

['all-mini_emb.npy', 'all-mpnet-base-v2_emb.npy', 'all-distilroberta-v1_emb.npy', 'bge_emb.npy', 'mpnet_distilled_regression.npy']


In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def compute_coherence(topic_model, docs):
    # Extract topic words for coherence calculation
    topics = topic_model.get_topics()
    topic_words = [[word for word, _ in topic_model.get_topic(topic_id)] for topic_id in topics.keys() if topic_id != -1]

    texts = [doc.split() for doc in docs]
    dictionary = Dictionary(texts)
    # Filter out words not in the dictionary
    topic_words = [[word for word in topic if word in dictionary.token2id] for topic in topic_words]

    cm = CoherenceModel(topics=topic_words, texts=texts, dictionary=dictionary, coherence='c_npmi')
    return cm.get_coherence()

from sklearn.metrics import silhouette_score

def compute_silhouette(topic_model, embeddings):
    topics = topic_model.topics_
    unique_topics = np.unique([t for t in topics if t != -1])
    mask = np.isin(topics, unique_topics)
    if len(unique_topics) < 2: # Check if there are at least two unique topics
      return 0.0 # Return 0 or handle as appropriate for no valid clusters
    return silhouette_score(embeddings[mask], np.array(topics)[mask])

def gini(array):
    array = np.sort(np.array(array))
    n = len(array)
    if np.sum(array) == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return ((np.sum((2 * index - n - 1) * array)) / (n * np.sum(array)))

In [ ]:
for file in list_of_files:
    print(file)
    time_start = time.time()

    embeddings = np.load(f'/content/drive/MyDrive/V_model/{file}')

    # Define models before use
    hdbscan_model = HDBSCAN(min_cluster_size=20, min_samples=15, metric='euclidean',
                            cluster_selection_method='eom', prediction_data=True)
    umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2),
                                       token_pattern=r"(?u)\b\w{3,}\b")

    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=30,
        nr_topics=50,
        top_n_words=10,
        verbose=False
    )

    topics, probs = topic_model.fit_transform(sentences, embeddings)

    coherence = compute_coherence(topic_model, sentences)
    silhouette = compute_silhouette(topic_model, embeddings)

    topic_info = topic_model.get_topic_info()
    error_size = topic_info['Count'].iloc[0]

    topic_info_filtered = topic_info[topic_info["Topic"] != -1]
    top_df = topic_info_filtered.nlargest(20, "Count").copy()

    topic20_size = 0
    if not top_df.empty and len(top_df) >= 20:
        topic20_size = top_df.iloc[19]['Count']

    ngram_value = 0
    gini_score = 0.0
    if not top_df.empty:
        topic_counts = top_df["Count"].values
        gini_score = gini(topic_counts)
        for topic_id in top_df['Topic'][:20]:
            for word, _ in topic_model.get_topic(topic_id):
                if word in term_freq_dict:
                    ngram_value += term_freq_dict[word]

    time_end = time.time()
    time_cost = (time_end - time_start)/60
    # Append metrics
    error_sizes.append(error_size)
    topic20_sizes.append(topic20_size)
    ngram_values.append(round(ngram_value / total_term_freq, 2))
    gini_scores.append(gini_score)
    model_names.append(file)
    coherences.append(round(coherence, 2))
    silhouettes.append(round(silhouette, 2))
    time_costs.append(round(time_cost, 2))

    # Clean up
    del topic_model, hdbscan_model, umap_model
    gc.collect()

all-mini_emb.npy
all-mpnet-base-v2_emb.npy
all-distilroberta-v1_emb.npy
bge_emb.npy
mpnet_distilled_regression.npy


In [ ]:
import pandas as pd

results = {
    "model_names": model_names,
    "error_sizes": error_sizes,
    "topic20_sizes": topic20_sizes,
    "ngram_values": ngram_values,
    "gini_scores": gini_scores,
    "coherences": coherences,
    "silhouettes": silhouettes,
    "time costs (min)": time_costs
}
df = pd.DataFrame(results)
df.head()

,model_names,error_sizes,topic20_sizes,ngram_values,gini_scores,coherences,silhouettes,time costs (min)
0,all-mini_emb.npy,60277,520,0.16,0.528517,0.06,0.00,10.96
1,all-mpnet-base-v2_emb.npy,62729,584,0.16,0.516172,0.09,0.00,10.22
2,all-distilroberta-v1_emb.npy,54131,922,0.19,0.390752,0.06,0.00,11.63
3,bge_emb.npy,59266,59,0.13,0.866747,NaN,-0.08,10.91
4,mpnet_distilled_regression.npy,63041,384,0.16,0.705805,0.26,-0.04,10.84


In [ ]:
df.to_csv('/content/drive/MyDrive/Bertopic/Data/emb_compare_results.csv', index=False)